## Transform II: Überschwemmungen transformieren

In diesem Abschnitt werden alle geforderten Transformationsschritte umgesetzt, aber die beiden Datensätze werden noch nicht miteinander verbunden. EMDAT wird fachlich auf Überschwemmungen im Mittelmeerraum vorbereitet, Sea-Level wird separat als Zeitreihe vorbereitet. Die eigentliche Zusammenführung der Datensätze passiert erst später.

### Entscheid: Long oder Wide?

| Datensatz / Zwischentabelle | Gewähltes Format | Grund |
|---|---|---|
| `emdat_flood` | **Wide** | Eine Zeile ist ein Flood-Ereignis; Spalten enthalten Land, Datum, Subtyp und Impact. |
| `flood_subtype_long` | **Long** | Gut für Gruppierungen und Visualisierungen nach Flood-Subtyp. |
| `flood_subtype_wide` | **Wide** | Gut für spätere Modellfeatures, weil jeder Subtyp eine eigene Spalte hat. |
| `sea_level_ts` | **Long/Zeitreihe** | Eine Zeile pro Zeitpunkt ist ideal für Resampling, Lag/Lead und Rolling Windows. |
| `sea_metric_long` | **Long** | Mehrere Sea-Level-Metriken werden untereinander gestellt, ohne mit EMDAT zu joinen. |

In [3]:
import logging
import numpy as np
from myproj.transform.trim_data import filter_before_sea_level_start

pd.set_option("display.max_columns", 60)

SEA_VALUE = "MSL_filtered_GIA_corrected_adjusted"
SEA_TREND = "trend_MSL_filtered_GIA_corrected_adjusted"

MED_ISO = [
    "ALB", "DZA", "BIH", "HRV", "CYP", "EGY", "FRA", "GRC", "ISR", "ITA", "LBN",
    "LBY", "MLT", "MNE", "MAR", "PSE", "SVN", "ESP", "SYR", "TUN", "TUR",
]


def make_event_date(df, prefix):
    """Create a date from EMDAT year/month/day columns and keep missingness transparent."""
    return pd.to_datetime(
        pd.DataFrame(
            {
                "year": pd.to_numeric(df[f"{prefix} Year"], errors="coerce"),
                "month": pd.to_numeric(df[f"{prefix} Month"], errors="coerce").fillna(1),
                "day": pd.to_numeric(df[f"{prefix} Day"], errors="coerce").fillna(1),
            }
        ),
        errors="coerce",
    )


def date_quality(df, prefix):
    month_missing = df[f"{prefix} Month"].isna()
    day_missing = df[f"{prefix} Day"].isna()
    return np.select(
        [month_missing & day_missing, month_missing, day_missing],
        ["month_and_day_imputed", "month_imputed", "day_imputed"],
        default="complete",
    )


def season_from_month(month):
    if pd.isna(month):
        return "unknown"
    month = int(month)
    if month in [12, 1, 2]:
        return "winter"
    if month in [3, 4, 5]:
        return "spring"
    if month in [6, 7, 8]:
        return "summer"
    return "autumn"


def standard_scale(series):
    x = pd.to_numeric(series, errors="coerce").astype(float)
    std = x.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(np.nan, index=x.index)
    return (x - x.mean()) / std


def minmax_scale(series):
    x = pd.to_numeric(series, errors="coerce").astype(float)
    value_range = x.max() - x.min()
    if value_range == 0 or pd.isna(value_range):
        return pd.Series(np.nan, index=x.index)
    return (x - x.min()) / value_range


def box_cox_transform(series, lmbda=0.0):
    """Box-Cox needs positive values; shifting by min + 1 preserves ordering."""
    x = pd.to_numeric(series, errors="coerce").astype(float)
    x_positive = x - x.min(skipna=True) + 1
    if np.isclose(lmbda, 0):
        transformed = np.log(x_positive)
    else:
        transformed = (np.power(x_positive, lmbda) - 1) / lmbda
    return pd.Series(transformed, index=x.index)


def yeo_johnson_transform(series, lmbda=0.5):
    """Yeo-Johnson also works with zero and negative values."""
    x = pd.to_numeric(series, errors="coerce").astype(float)
    transformed = pd.Series(np.nan, index=x.index, dtype="float64")
    valid = x.notna()
    pos = valid & (x >= 0)
    neg = valid & (x < 0)

    if np.isclose(lmbda, 0):
        transformed.loc[pos] = np.log1p(x.loc[pos])
    else:
        transformed.loc[pos] = (np.power(x.loc[pos] + 1, lmbda) - 1) / lmbda

    if np.isclose(lmbda, 2):
        transformed.loc[neg] = -np.log1p(-x.loc[neg])
    else:
        transformed.loc[neg] = -(
            (np.power(-x.loc[neg] + 1, 2 - lmbda) - 1) / (2 - lmbda)
        )
    return transformed

ModuleNotFoundError: No module named 'myproj'

### EMDAT Floods: filtern, sortieren, kombinieren, Strings und abgeleitete Variablen

Hier wird nur EMDAT bearbeitet. Die Filter werden kombiniert, die Tabelle wird sortiert, Stringspalten werden bereinigt, aus bestehenden Variablen werden neue Variablen abgeleitet und der Zeitraum wird auf die Sea-Level-Abdeckung begrenzt.

In [ ]:
emdat_transform = filter_before_sea_level_start(emdat).copy()
emdat_transform["start_date"] = make_event_date(emdat_transform, "Start")
emdat_transform["end_date"] = make_event_date(emdat_transform, "End")
emdat_transform["start_date_quality"] = date_quality(emdat_transform, "Start")
emdat_transform["end_date_quality"] = date_quality(emdat_transform, "End")
emdat_transform["year_month"] = emdat_transform["start_date"].dt.to_period("M").dt.to_timestamp()
emdat_transform["season"] = emdat_transform["Start Month"].apply(season_from_month)
emdat_transform["event_duration_days"] = (
    (emdat_transform["end_date"] - emdat_transform["start_date"]).dt.days + 1
).clip(lower=1)
emdat_transform["has_coordinates"] = emdat_transform[["Latitude", "Longitude"]].notna().all(axis=1)
emdat_transform["affected_total_filled"] = pd.to_numeric(
    emdat_transform["Total Affected"], errors="coerce"
).fillna(0)
emdat_transform["deaths_total_filled"] = pd.to_numeric(
    emdat_transform["Total Deaths"], errors="coerce"
).fillna(0)
emdat_transform["impact_total"] = emdat_transform["affected_total_filled"] + emdat_transform["deaths_total_filled"]
emdat_transform["impact_log1p"] = np.log1p(emdat_transform["impact_total"])

sea_coverage_time = pd.to_datetime(sea_level["time"])
sea_coverage_start = sea_coverage_time.min()
sea_coverage_end = sea_coverage_time.max()

flood_text_mask = (
    emdat_transform["Disaster Type"].str.contains("flood", case=False, na=False)
    | emdat_transform["Disaster Subtype"].str.contains("flood", case=False, na=False)
)
mediterranean_mask = emdat_transform["ISO"].isin(MED_ISO)
has_valid_date_mask = emdat_transform["start_date"].notna()
within_sea_coverage_mask = emdat_transform["start_date"].between(sea_coverage_start, sea_coverage_end)

emdat_flood = (
    emdat_transform.loc[
        flood_text_mask & mediterranean_mask & has_valid_date_mask & within_sea_coverage_mask
    ]
    .copy()
    .sort_values(["start_date", "Country", "DisNo."])
    .reset_index(drop=True)
)

emdat_flood["country_clean"] = emdat_flood["Country"].str.replace(r"\s+", " ", regex=True).str.strip()
emdat_flood["flood_subtype_clean"] = (
    emdat_flood["Disaster Subtype"]
    .fillna("unknown")
    .str.lower()
    .str.replace(r"\s*\([^)]*\)", "", regex=True)
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
emdat_flood["origin_clean"] = (
    emdat_flood["Origin"]
    .fillna("unknown")
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
emdat_flood["flood_year"] = emdat_flood["start_date"].dt.year
emdat_flood["flood_month"] = emdat_flood["start_date"].dt.month
emdat_flood["affected_per_death"] = np.where(
    emdat_flood["deaths_total_filled"] > 0,
    emdat_flood["affected_total_filled"] / emdat_flood["deaths_total_filled"],
    np.nan,
)

disno_parts = emdat_flood["DisNo."].str.extract(
    r"^(?P<disno_year>\d{4})-(?P<disno_sequence>\d{4})-(?P<disno_iso>[A-Z]{3})$"
)
disno_parts["disno_year"] = pd.to_numeric(disno_parts["disno_year"], errors="coerce")
disno_parts["disno_sequence"] = pd.to_numeric(disno_parts["disno_sequence"], errors="coerce")
emdat_flood = pd.concat([emdat_flood, disno_parts], axis=1)

print(f"Flood-Ereignisse im Mittelmeerraum: {len(emdat_flood):,}")
print(f"Sea-Level-Abdeckung: {sea_coverage_start.date()} bis {sea_coverage_end.date()}")
print(f"Flood-Zeitraum nach Ausrichtung: {emdat_flood['start_date'].min().date()} bis {emdat_flood['start_date'].max().date()}")
display(
    emdat_flood[
        [
            "DisNo.", "start_date", "Country", "country_clean", "Disaster Type",
            "Disaster Subtype", "flood_subtype_clean", "origin_clean", "impact_total",
            "impact_log1p", "disno_year", "disno_sequence", "disno_iso",
        ]
    ].head(10)
)

### EMDAT Floods: gruppieren, aggregieren und Long/Wide umwandeln

Die Flood-Ereignisse werden nach Monat, Land und Subtyp aggregiert. Zusätzlich wird eine Subtyp-Tabelle von Long nach Wide transformiert und danach wieder von Wide nach Long zurückgeführt.

In [ ]:
flood_subtype_long = (
    emdat_flood.groupby(["year_month", "flood_subtype_clean"])
    .size()
    .reset_index(name="n_flood_events")
)

flood_subtype_wide = (
    flood_subtype_long.pivot_table(
        index="year_month",
        columns="flood_subtype_clean",
        values="n_flood_events",
        fill_value=0,
    )
    .add_prefix("n_flood_subtype_")
    .reset_index()
)
flood_subtype_wide.columns.name = None

flood_subtype_long_again = flood_subtype_wide.melt(
    id_vars="year_month",
    var_name="flood_subtype",
    value_name="n_flood_events",
)
flood_subtype_long_again["flood_subtype"] = flood_subtype_long_again["flood_subtype"].str.replace(
    r"^n_flood_subtype_", "", regex=True
)

flood_monthly_base = (
    emdat_flood.groupby("year_month")
    .agg(
        flood_events=("DisNo.", "nunique"),
        flood_countries=("ISO", "nunique"),
        flood_affected_total=("affected_total_filled", "sum"),
        flood_deaths_total=("deaths_total_filled", "sum"),
        flood_median_duration_days=("event_duration_days", "median"),
        flood_coordinates_share=("has_coordinates", "mean"),
        flood_primary_subtype=("flood_subtype_clean", lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
    )
    .reset_index()
)

# Kombination innerhalb des EMDAT-Flood-Datensatzes, kein Join mit Sea-Level.
flood_monthly = flood_monthly_base.merge(flood_subtype_wide, on="year_month", how="left")
flood_subtype_cols = [c for c in flood_monthly.columns if c.startswith("n_flood_subtype_")]
flood_monthly[flood_subtype_cols] = flood_monthly[flood_subtype_cols].fillna(0).astype(int)
flood_monthly["affected_per_flood"] = np.where(
    flood_monthly["flood_events"] > 0,
    flood_monthly["flood_affected_total"] / flood_monthly["flood_events"],
    0,
)
flood_monthly["flood_death_share"] = np.where(
    flood_monthly["flood_affected_total"] > 0,
    flood_monthly["flood_deaths_total"] / flood_monthly["flood_affected_total"],
    0,
)

flood_by_country_year = (
    emdat_flood.groupby(["flood_year", "country_clean"])
    .agg(
        flood_events=("DisNo.", "nunique"),
        affected_total=("affected_total_filled", "sum"),
        deaths_total=("deaths_total_filled", "sum"),
    )
    .reset_index()
    .sort_values(["flood_year", "flood_events", "affected_total"], ascending=[True, False, False])
)

print(f"Flood-Monatsaggregation: {flood_monthly.shape[0]:,} Monate mit Floods")
print(f"Long -> Wide -> Long: {flood_subtype_long.shape} -> {flood_subtype_wide.shape} -> {flood_subtype_long_again.shape}")
display(flood_monthly.head())
display(flood_by_country_year.head(10))
display(flood_subtype_long_again.query("n_flood_events > 0").head(10))

### EMDAT Floods: Date Range, Lag/Lead, Moving Window und Reskalierung

Die Vollständigkeit wird zuerst an den tatsächlich beobachteten Flood-Monaten geprüft. Erst danach wird eine vollständige Monats-Range aufgebaut; darauf entstehen Lag-/Lead-Variablen, rollierende Fenster und vier Skalierungen.

In [ ]:
observed_flood_months = pd.DatetimeIndex(flood_monthly["year_month"].sort_values().unique())
expected_flood_months = pd.date_range(
    observed_flood_months.min(),
    observed_flood_months.max(),
    freq="MS",
)
missing_flood_months = expected_flood_months.difference(observed_flood_months)

flood_monthly_ts = (
    pd.DataFrame({"year_month": expected_flood_months})
    .merge(flood_monthly, on="year_month", how="left")
    .sort_values("year_month")
    .reset_index(drop=True)
)

flood_fill_cols = [
    "flood_events", "flood_countries", "flood_affected_total", "flood_deaths_total",
    "flood_median_duration_days", "flood_coordinates_share", "affected_per_flood", "flood_death_share",
] + [c for c in flood_monthly_ts.columns if c.startswith("n_flood_subtype_")]
for col in flood_fill_cols:
    if col in flood_monthly_ts.columns:
        flood_monthly_ts[col] = flood_monthly_ts[col].fillna(0)

flood_monthly_ts["flood_primary_subtype"] = flood_monthly_ts["flood_primary_subtype"].fillna("none")
flood_monthly_ts["has_flood"] = flood_monthly_ts["flood_events"].gt(0).astype(int)
flood_monthly_ts["flood_events_lag_1m"] = flood_monthly_ts["flood_events"].shift(1)
flood_monthly_ts["flood_events_lead_1m"] = flood_monthly_ts["flood_events"].shift(-1)
flood_monthly_ts["flood_affected_lag_1m"] = flood_monthly_ts["flood_affected_total"].shift(1)
flood_monthly_ts["flood_affected_lead_1m"] = flood_monthly_ts["flood_affected_total"].shift(-1)
flood_monthly_ts["flood_events_roll_3m_mean"] = flood_monthly_ts["flood_events"].rolling(window=3, min_periods=1).mean()
flood_monthly_ts["flood_events_roll_12m_sum"] = flood_monthly_ts["flood_events"].rolling(window=12, min_periods=1).sum()
flood_monthly_ts["flood_affected_roll_12m_sum"] = flood_monthly_ts["flood_affected_total"].rolling(window=12, min_periods=1).sum()

flood_scaled = flood_monthly_ts[["year_month", "flood_events", "flood_affected_total"]].copy()
flood_scaled["flood_events_z"] = standard_scale(flood_scaled["flood_events"])
flood_scaled["flood_events_minmax"] = minmax_scale(flood_scaled["flood_events"])
flood_scaled["flood_affected_boxcox_z"] = standard_scale(
    box_cox_transform(flood_scaled["flood_affected_total"], lmbda=0.0)
)
flood_scaled["flood_affected_yeojohnson_z"] = standard_scale(
    yeo_johnson_transform(flood_scaled["flood_affected_total"], lmbda=0.5)
)

print(f"Beobachtete Flood-Monate: {len(observed_flood_months):,}")
print(f"Erwartete Flood-Monate im ausgerichteten Zeitraum: {len(expected_flood_months):,}")
print(f"Fehlende Monate in der Flood-Zeitreihe: {len(missing_flood_months):,}")
display(
    flood_monthly_ts[
        [
            "year_month", "flood_events", "flood_events_lag_1m", "flood_events_lead_1m",
            "flood_events_roll_3m_mean", "flood_events_roll_12m_sum", "flood_affected_total",
            "flood_affected_roll_12m_sum",
        ]
    ].head(15)
)
display(flood_scaled.head(10))

NameError: name 'flood_monthly' is not defined

### Sea-Level separat transformieren

Der Sea-Level-Datensatz wird nicht mit EMDAT zusammengeführt. Er wird separat sortiert, resampled, in Long/Wide-Formate umgewandelt, skaliert und als Zeitreihe mit Lag/Lead und Rolling Windows vorbereitet.

In [ ]:
sea_level_ts = sea_level.copy()
sea_level_ts["time"] = pd.to_datetime(sea_level_ts["time"])
sea_level_ts = sea_level_ts.sort_values("time").reset_index(drop=True)
sea_level_ts["year_month"] = sea_level_ts["time"].dt.to_period("M").dt.to_timestamp()
sea_level_ts["year"] = sea_level_ts["time"].dt.year
sea_level_ts["month"] = sea_level_ts["time"].dt.month
sea_level_ts["msl_anomaly_mm"] = sea_level_ts[SEA_VALUE] * 10
sea_level_ts["msl_minus_trend_cm"] = sea_level_ts[SEA_VALUE] - sea_level_ts[SEA_TREND]

sea_metric_long = sea_level_ts.melt(
    id_vars=["time", "year_month"],
    value_vars=[SEA_VALUE, SEA_TREND, "msl_anomaly_mm", "msl_minus_trend_cm"],
    var_name="sea_metric",
    value_name="sea_value",
)
sea_metric_wide_again = (
    sea_metric_long.pivot_table(
        index=["time", "year_month"],
        columns="sea_metric",
        values="sea_value",
    )
    .reset_index()
)
sea_metric_wide_again.columns.name = None

sea_monthly = (
    sea_level_ts.set_index("time")
    .resample("MS")
    .agg(
        msl_mean_cm=(SEA_VALUE, "mean"),
        msl_min_cm=(SEA_VALUE, "min"),
        msl_max_cm=(SEA_VALUE, "max"),
        trend_mean_cm=(SEA_TREND, "mean"),
        n_days=(SEA_VALUE, "size"),
    )
    .rename_axis("year_month")
    .reset_index()
)
sea_monthly["msl_range_cm"] = sea_monthly["msl_max_cm"] - sea_monthly["msl_min_cm"]
sea_monthly["msl_mean_lag_1m"] = sea_monthly["msl_mean_cm"].shift(1)
sea_monthly["msl_mean_lead_1m"] = sea_monthly["msl_mean_cm"].shift(-1)
sea_monthly["msl_mean_roll_3m"] = sea_monthly["msl_mean_cm"].rolling(window=3, min_periods=1).mean()
sea_monthly["msl_mean_roll_12m"] = sea_monthly["msl_mean_cm"].rolling(window=12, min_periods=1).mean()

sea_month_range = pd.date_range(sea_monthly["year_month"].min(), sea_monthly["year_month"].max(), freq="MS")
missing_sea_months = sea_month_range.difference(sea_monthly["year_month"])

sea_scaled = sea_monthly[["year_month", "msl_mean_cm", "msl_range_cm"]].copy()
sea_scaled["msl_mean_cm_z"] = standard_scale(sea_scaled["msl_mean_cm"])
sea_scaled["msl_mean_cm_minmax"] = minmax_scale(sea_scaled["msl_mean_cm"])
sea_scaled["msl_range_boxcox_z"] = standard_scale(box_cox_transform(sea_scaled["msl_range_cm"], lmbda=0.0))
sea_scaled["msl_mean_yeojohnson_z"] = standard_scale(yeo_johnson_transform(sea_scaled["msl_mean_cm"], lmbda=0.5))

print(f"Sea-Level Tage: {len(sea_level_ts):,}")
print(f"Sea-Level Monate: {len(sea_monthly):,}; fehlende Monate: {len(missing_sea_months):,}")
print(f"Sea-Level Wide -> Long -> Wide: {sea_level_ts[[SEA_VALUE, SEA_TREND]].shape} -> {sea_metric_long.shape} -> {sea_metric_wide_again.shape}")
display(sea_monthly.head())
display(sea_metric_long.head(8))
display(sea_scaled.head(10))

### Logs in Funktionen

Die folgenden Funktionen implementieren Logging für die Flood-Transformation und die Sea-Level-Transformation.

In [ ]:
logger = logging.getLogger("daw.transform.flood")
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter("%(levelname)s:%(name)s:%(message)s"))
    logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False


def build_flood_monthly_with_logs(flood_events):
    logger.info("Starte Flood-Aggregation: %s Ereignisse", len(flood_events))
    required_cols = {"year_month", "DisNo.", "affected_total_filled", "deaths_total_filled", "flood_subtype_clean"}
    missing_cols = required_cols.difference(flood_events.columns)
    if missing_cols:
        logger.error("Fehlende Flood-Spalten: %s", sorted(missing_cols))
        raise ValueError(f"Fehlende Flood-Spalten: {sorted(missing_cols)}")

    result = (
        flood_events.groupby("year_month")
        .agg(
            flood_events=("DisNo.", "nunique"),
            flood_affected_total=("affected_total_filled", "sum"),
            flood_deaths_total=("deaths_total_filled", "sum"),
            flood_primary_subtype=("flood_subtype_clean", lambda s: s.mode().iat[0] if not s.mode().empty else np.nan),
        )
        .reset_index()
    )
    logger.info("Flood-Aggregation fertig: %s Monate", len(result))
    return result


def build_sea_monthly_with_logs(sea_daily):
    logger.info("Starte Sea-Level-Aggregation: %s Tageswerte", len(sea_daily))
    required_cols = {"time", SEA_VALUE, SEA_TREND}
    missing_cols = required_cols.difference(sea_daily.columns)
    if missing_cols:
        logger.error("Fehlende Sea-Level-Spalten: %s", sorted(missing_cols))
        raise ValueError(f"Fehlende Sea-Level-Spalten: {sorted(missing_cols)}")

    result = (
        sea_daily.set_index("time")
        .resample("MS")
        .agg(msl_mean_cm=(SEA_VALUE, "mean"), trend_mean_cm=(SEA_TREND, "mean"))
        .rename_axis("year_month")
        .reset_index()
    )
    logger.info("Sea-Level-Aggregation fertig: %s Monate", len(result))
    return result


flood_monthly_logged = build_flood_monthly_with_logs(emdat_flood)
sea_monthly_logged = build_sea_monthly_with_logs(sea_level_ts)

display(flood_monthly_logged.head())
display(sea_monthly_logged.head())

NameError: name 'logging' is not defined